# Export the full document-classification model to Hugging Face

This notebook loads the LoRA adapter from this GitHub project, merges it into `google/gemma-2-2b-it`, and uploads the resulting standalone model and tokenizer to a Hugging Face model repository.

## Requirements and cautions

- Use a Colab GPU with enough memory for the base model and merged weights. A T4 may work, but merging can require additional RAM/VRAM; use a high-memory runtime if available.
- Accept the [Gemma 2 2B terms](https://huggingface.co/google/gemma-2-2b-it) before downloading the gated base model.
- Add a Hugging Face token with **write** access to Colab Secrets as `HF_TOKEN`. It is only read at runtime and is not written into the notebook.
- Set the destination repository ID below. The repo is created **private by default**. Before making a Gemma-derived model public, check and follow the applicable Gemma terms and Hugging Face requirements.
- The adapter was trained for document format and administrative purpose classification. Merging it does not add training for other extracted metadata fields.


In [ ]:
%pip -q install transformers peft accelerate huggingface_hub

## Configure source and destination

The adapter path is relative to the GitHub repository root. Change these settings if your repository, branch, or folder differs.

In [ ]:
from pathlib import Path
import subprocess

GITHUB_REPO_URL = "https://github.com/edla4eva/capstone.git"
GITHUB_BRANCH = "main"
ADAPTER_PATH = Path("capstone/gemma-instruction-adapter")
REPO_DIR = Path("/content/project")

HF_MODEL_REPO_ID = "YOUR_USERNAME/gemma-admin-document-classifier"
MAKE_REPO_PRIVATE = True
BASE_MODEL_ID = "google/gemma-2-2b-it"

if "YOUR_USERNAME" in HF_MODEL_REPO_ID:
    raise ValueError("Set HF_MODEL_REPO_ID to your Hugging Face username and model repository name.")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(REPO_DIR)],
        check=True,
    )

ADAPTER_DIR = REPO_DIR / ADAPTER_PATH
if not (ADAPTER_DIR / "adapter_model.safetensors").is_file():
    raise FileNotFoundError(
        f"LoRA weights not found in {ADAPTER_DIR}. Check GITHUB_REPO_URL, GITHUB_BRANCH, and ADAPTER_PATH."
    )

print(f"Adapter: {ADAPTER_DIR}")
print(f"Hugging Face destination: {HF_MODEL_REPO_ID} (private={MAKE_REPO_PRIVATE})")

## Authenticate

Use a Hugging Face write token: it needs permission to download the gated base model and create/upload the destination model repository.

In [ ]:
from huggingface_hub import HfApi

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN", default=None)
except (ImportError, KeyError):
    hf_token = None

if not hf_token:
    from getpass import getpass
    hf_token = getpass("Hugging Face write token (input hidden): ")
if not hf_token:
    raise ValueError("A Hugging Face token with write access is required.")

api = HfApi(token=hf_token)
api.create_repo(
    repo_id=HF_MODEL_REPO_ID,
    repo_type="model",
    private=MAKE_REPO_PRIVATE,
    exist_ok=True,
)
print("Destination repository is ready.")

## Load, merge, and upload

The model is loaded in FP16 to reduce memory use. `merge_and_unload()` folds the LoRA weights into the base-model weights so inference no longer needs PEFT or the separate adapter.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Colab GPU runtime before loading and merging the model.")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=hf_token)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    token=hf_token,
    low_cpu_mem_usage=True,
)
adapter_model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
merged_model = adapter_model.merge_and_unload(safe_merge=True)
merged_model.eval()

print("LoRA adapter merged into the base model.")

## Upload the standalone model and tokenizer

This uploads a complete model (larger than the adapter-only folder). Keep the Colab session active until both uploads finish.

In [ ]:
merged_model.push_to_hub(
    HF_MODEL_REPO_ID,
    token=hf_token,
    private=MAKE_REPO_PRIVATE,
    safe_serialization=True,
    max_shard_size="2GB",
)
tokenizer.push_to_hub(HF_MODEL_REPO_ID, token=hf_token)
print(f"Full model uploaded: https://huggingface.co/{HF_MODEL_REPO_ID}")

## Verify by reloading the uploaded full model

This check confirms that the destination contains a complete model loadable without PEFT. It downloads the uploaded weights again.

In [ ]:
del adapter_model, base_model, merged_model
torch.cuda.empty_cache()

check_tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_REPO_ID, token=hf_token)
check_model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_REPO_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    token=hf_token,
)
print("Verified: the uploaded repository loads as a standalone model.")
del check_model, check_tokenizer
torch.cuda.empty_cache()
hf_token = None